In [ ]:
# PASO 5 — Codificación de variables categóricas (versión final)


cols_redundantes = [
    'primaryTumorType', 'primaryTumorSubType', 'primaryTumorSubLocation',
    'primaryTumorLocation', 'primaryTumorExtraDetails',
    'tissue_type', 'cancer_type', 'cancer_type_code', 'Mixed'
]
df_modelo = df_final.drop(columns=cols_redundantes)
df_modelo = df_modelo.drop(columns=['es_ALT_positivo'])

df_encoded = pd.get_dummies(
    df_modelo,
    columns=['Mixed_def', 'tissue_group', 'msi_status', 'hr_status'],
    drop_first=True
)

print(f"Dataset final: {df_encoded.shape[0]} pacientes, {df_encoded.shape[1]} columnas")
df_encoded.to_csv("../datos_modelo_final.csv", index=False)
print("Guardado: datos_modelo_final.csv")

In [ ]:

# PASO 4 — Integrar columnas desde Metadatos_to_ALV.xlsx


meta = pd.read_excel("../Metadatos_to_ALV.xlsx")


cols_meta = ['Patient_ID_IF_mod', 'msi_status', 'hr_status']

df_final = df_final.merge(meta[cols_meta], on='Patient_ID_IF_mod', how='left')

  # Eliminar los pacientes sin cobertura en el archivo de metadatos
df_final = df_final.dropna(subset=['msi_status', 'hr_status']).reset_index(drop=True)

  # Un valor de msi_status era el nombre completo de la columna — corregirlo
df_final['msi_status'] = df_final['msi_status'].replace({
    'Microsatellite instability (MSI) status. Also known as mismatch repair deficiency (MMRD)': 'MSI'
})

print(f"Pacientes tras integrar metadatos: {len(df_final)}")
print(f"ALT-positivos conservados:         {(df_final[TARGET] > 0.2).sum()}")
print(f"NaN total:                         {df_final.isna().sum().sum()}")

  # Sobreescribir el CSV limpio con la versión que incluye los metadatos
df_final.to_csv("../datos_limpios_modelo_a.csv", index=False)
print("Guardado: datos_limpios_modelo_a.csv (con msi_status y hr_status)")

In [ ]:

# CODIFICACIÓN DE VARIABLES CATEGÓRICAS

import pandas as pd

df = pd.read_csv("../datos_limpios_modelo_a.csv")
TARGET = 'Tumor_Telomere fusion rate (Gt-end/coverage*purity)'

# Columnas categóricas redundantes o con demasiados "0" → eliminar
cols_redundantes = [
    'primaryTumorType', 'primaryTumorSubType', 'primaryTumorSubLocation',
    'primaryTumorLocation', 'primaryTumorExtraDetails',
    'tissue_type', 'cancer_type', 'cancer_type_code', 'Mixed'
]
df = df.drop(columns=cols_redundantes)

print(f"Columnas antes de codificar: {df.shape[1]}")
print(df.dtypes)

  # One-hot encoding de Mixed_def y tissue_group
  # drop_first=True elimina una categoría por columna para evitar trampa de la variable dummy
df_encoded = pd.get_dummies(df, columns=['Mixed_def', 'tissue_group'], drop_first=True)

  # Quitar la bandera auxiliar antes de guardar (no es una feature)
df_encoded = df_encoded.drop(columns=['es_ALT_positivo'])

print(f"Columnas después de codificar: {df_encoded.shape[1]}")
print(f"Filas: {df_encoded.shape[0]}")
print(f"\nPrimeras columnas nuevas de Mixed_def:")
print([c for c in df_encoded.columns if c.startswith('Mixed_def')][:5])

  # Guardar dataset final para el modelo
df_encoded.to_csv("../datos_modelo_final.csv", index=False)
print("Guardado: datos_modelo_final.csv")
print(f"\nResumen final:")
print(f"  Filas (pacientes): {df_encoded.shape[0]}")
print(f"  Columnas (features + target): {df_encoded.shape[1]}")
print(f"  Columnas numéricas de fusión: 6")
print(f"  Columnas codificadas Mixed_def: {sum(c.startswith('Mixed_def') for c in df_encoded.columns)}")
print(f"  Columnas codificadas tissue_group: {sum(c.startswith('tissue_group') for c in df_encoded.columns)}")

In [ ]:
# DATASET FINAL — Modelo A (primera iteración)


TARGET = 'Tumor_Telomere fusion rate (Gt-end/coverage*purity)'

cols_modelo_a = [
    'Patient_ID_IF_mod',
    TARGET,
    'Tumor_TF rate INWARDS',
    'Tumor_TF rate OUTWARDS',
    'Blood_Telomere fusion rate (Gt-end/coverage*purity)',
    'Blood_TF rate INWARDS',
    'Blood_TF rate OUTWARDS',
    'primaryTumorType',
    'primaryTumorSubType',
    'primaryTumorSubLocation',
    'primaryTumorLocation',
    'primaryTumorExtraDetails',
    'tissue_type',
    'tissue_group',
    'cancer_type',
    'cancer_type_code',
    'Mixed_def',
    'Mixed',
]

df_final = df[cols_modelo_a].dropna().reset_index(drop=True)

# Columna auxiliar para análisis (no entra al modelo)
df_final['es_ALT_positivo'] = df_final[TARGET] > 0.2

print("=== DATASET FINAL ===")
print(f"Pacientes:       {len(df_final)}")
print(f"Columnas:        {df_final.shape[1] - 1}  (sin contar la bandera)")
print(f"ALT-positivos:   {df_final['es_ALT_positivo'].sum()}")
print(f"NaN total:       {df_final.drop(columns='es_ALT_positivo').isna().sum().sum()}")

  # Guardar el dataset limpio para no repetir este proceso
df_final.to_csv("../datos_limpios_modelo_a.csv", index=False)
print("Guardado: datos_limpios_modelo_a.csv")

In [ ]:
# PASO 3 — Marcar pacientes intocables y contar supervivientes


TARGET = 'Tumor_Telomere fusion rate (Gt-end/coverage*purity)'

# Marcar los 338 pacientes ALT-positivos
df_limpio['es_ALT_positivo'] = df_limpio[TARGET] > 0.2

print(f"Pacientes ALT-positivos (target > 0.2): {df_limpio['es_ALT_positivo'].sum()}")

  # Ver cuántos quedan si eliminamos filas con NaN (sin contar la columna bandera)
cols_para_modelo = [c for c in df_limpio.columns if c != 'es_ALT_positivo']
df_sin_nan = df_limpio.dropna(subset=cols_para_modelo)

print(f"\nTotal pacientes sin NaN: {len(df_sin_nan)}")
print(f"ALT-positivos conservados: {df_sin_nan['es_ALT_positivo'].sum()}")
print(f"ALT-positivos perdidos: {df_limpio['es_ALT_positivo'].sum() - df_sin_nan['es_ALT_positivo'].sum()}")

In [ ]:
# PASO 2 — Crear feature de días hasta biopsia


df_limpio['dias_hasta_biopsia'] = (
    pd.to_datetime(df_limpio['biopsyDate']) -
    pd.to_datetime(df_limpio['treatmentStartDate'])
).dt.days

print("Valores de dias_hasta_biopsia:")
print(df_limpio['dias_hasta_biopsia'].describe())
print(f"\nNulos en esta nueva columna: {df_limpio['dias_hasta_biopsia'].isna().sum()}")

In [ ]:
# PASO 1 — Eliminar columnas claramente inútiles


  # Columna constante (inútil)
cols_eliminar = ['Patient']

  # Todos los treatments del 2 en adelante (>77% nulos)
  # treatment2, treatment2.1, treatment2.2, treatment3... hasta treatment10.2
for i in range(2, 11):
    for sufijo in ['', '.1', '.2']:
        col = f'treatment{i}{sufijo}'
        if col in df.columns:
            cols_eliminar.append(col)

  # Columnas con >50% nulos que no aportan al modelo inicial
cols_eliminar += [
    'Tumor_TF/Blood TF',    # derivable de otras, 56% nulos
    'responseDate',          # 59% nulos
    'firstResponse',         # 62% nulos
    'responseMeasured',      # 62% nulos
    'deathDate',             # 63% nulos
    'cancer_subtype',        # 77% nulos
    'treatmentEndDate',      # 51% nulos
    'consolidatedTreatmentType',  # 50% nulos
]

df_limpio = df.drop(columns=cols_eliminar)
print(f"Antes: {df.shape[1]} columnas → Después: {df_limpio.shape[1]} columnas")
print(f"Eliminadas: {len(cols_eliminar)} columnas")

In [ ]:
 # Distribución del target real
target = df["Tumor_Telomere fusion rate (Gt-end/coverage*purity)"]

print("=== ESTADÍSTICAS DEL TARGET ===")
print(target.describe())
print(f"\nPacientes con valor 0 exacto:  {(target == 0).sum()}")
print(f"Pacientes con valor > 0.2:     {(target > 0.2).sum()}")
print(f"Pacientes con valor > 1:       {(target > 1).sum()}")
print(f"Pacientes con valor > 10:      {(target > 10).sum()}")

In [ ]:
# Ver el resumen COMPLETO de nulos, ordenado de más a menos
resumen_ordenado = resumen.sort_values("pct_nulos", ascending=False)
print(resumen_ordenado.to_string())  

In [ ]:
# En pandas las columnas son por nombre, no por letra

print("Primeras 5 columnas:", df.columns[:5].tolist())

# Y ver la distribución del target
df.iloc[:, 1].describe()  # iloc[:,1] = segunda columna = columna B

In [ ]:
# Nombre, tipo de dato, cuántos valores NO son NaN, y cuántos SÍ faltan
resumen = pd.DataFrame({
    "columna":   df.columns,
    "tipo":      df.dtypes.values,
    "no_nulos":  df.notna().sum().values,
    "nulos":     df.isna().sum().values,
    "pct_nulos": (df.isna().sum().values / len(df) * 100).round(1)
})
resumen

In [ ]:
!pip install seaborn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Rutas (ajusta si tu Jupyter arranca desde otra carpeta)
RUTA_PRINCIPAL = "../Tumor_blood_to_ALV_2.xlsx"
RUTA_METADATOS = "../Metadatos_to_ALV.xlsx"

# Cargamos SOLO Sheet1 (las demás son pruebas auxiliares)
df = pd.read_excel(RUTA_PRINCIPAL, sheet_name="Sheet1")

print(f"Filas: {df.shape[0]}")
print(f"Columnas: {df.shape[1]}")
df.head(3)